# NYC Yellow Taxi Trips (May – July 2026) — Programming for Data Science, Final Project

**Team members**
- Thierno DIALLO (team head)
- Joe VASEEKARAN (team member)
- Lisandru ZUCCARELLI (team member)
- Noé PAYAN (team member)

**GitHub repository:** https://github.com/diallothierno0223/programming_in_ds_taxi_study

**Dataset:** *NYC TLC Yellow Taxi Trip Records* — https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

This notebook follows the **KDD process** (Fayyad et al., 1996):
selection → pre-processing → transformation → data mining → interpretation / visualisation.
Every step is implemented as a documented function; all functions are called from the
`__main__` block at the end of the notebook, which also launches the Dash dashboard.

## About the dataset (human-readable description)

Every time a yellow taxi in New York City finishes a trip, the taximeter records it and sends it to the
**NYC Taxi & Limousine Commission (TLC)**. The TLC publishes these records monthly. We use the three
most recent months available: **May, June and July 2026** (one Parquet file per month), plus two
reference files that describe the 263 official *taxi zones* of the city.

Each row is **one taxi trip**. The dataset is *multidimensional*:

| Dimension | Columns | Meaning |
|---|---|---|
| **Temporal** (when) | `tpep_pickup_datetime`, `tpep_dropoff_datetime` | Start and end timestamp of the trip |
| **Spatial** (where) | `PULocationID`, `DOLocationID` | Pick-up / drop-off taxi zone (1–263), linked to `taxi_zone_lookup.csv` (zone & borough names) and to the shapefile `taxi_zones/` (polygons) |
| **Analytical** (what) | `trip_distance`, `passenger_count`, `fare_amount`, `tip_amount`, `tolls_amount`, `total_amount`, `payment_type`, `RatecodeID`, surcharges… | Characteristics of the trip: length, price, how it was paid, tip… |

Useful code meanings (from the TLC data dictionary):
- `payment_type`: 0 = flex fare, 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 5 = unknown, 6 = voided trip
- `RatecodeID`: 1 = standard, 2 = JFK, 3 = Newark, 4 = Nassau/Westchester, 5 = negotiated, 6 = group ride, 99 = unknown
- `VendorID`: technology provider of the taximeter
- Amounts are in US dollars, distances in miles. **Cash tips are not recorded** (only card tips are).

In [1]:
# Import libraries
import os                                   # paths / environment variables
import warnings                             # silence noisy library warnings
import numpy as np                          # numerical computing
import pandas as pd                         # tabular data manipulation
import geopandas as gpd                     # spatial data (taxi zone shapefile)
import plotly.express as px                 # interactive figures
import plotly.graph_objects as go           # low-level plotly figures
from mlxtend.frequent_patterns import apriori, association_rules   # frequent pattern mining
from statsmodels.tsa.holtwinters import ExponentialSmoothing       # forecasting
import statsmodels.formula.api as smf                               # regression (bonus: weather)
from sklearn.preprocessing import StandardScaler                    # feature scaling
from sklearn.cluster import KMeans                                  # spatial clustering
from sklearn.metrics import silhouette_score                        # clustering quality
from dash import Dash, html, dcc                                    # dashboard

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Global configuration
DATA_DIR = "data"                                               # folder with the raw files
OUTPUT_DIR = "outputs"                                          # folder for exported results
TRIP_FILES = [os.path.join(DATA_DIR, f"yellow_tripdata_2026-{m:02d}.parquet") for m in (5, 6, 7)]
ZONE_LOOKUP = os.path.join(DATA_DIR, "taxi_zone_lookup.csv")
ZONE_SHAPEFILE = os.path.join(DATA_DIR, "taxi_zones", "taxi_zones.shp")
WEATHER_FILE = os.path.join(DATA_DIR, "noaa_central_park_daily_2026.csv")   # external dataset (bonus)
# US federal holidays of the period (Memorial Day, Juneteenth, Independence Day + its observed Friday)
HOLIDAYS = pd.to_datetime(["2026-05-25", "2026-06-19", "2026-07-03", "2026-07-04"])
START, END = pd.Timestamp("2026-05-01"), pd.Timestamp("2026-08-01")  # study period
TEAM = ["Thierno DIALLO (team head)", "Joe VASEEKARAN", "Lisandru ZUCCARELLI", "Noé PAYAN"]
DATASET_NAME = "NYC TLC Yellow Taxi Trip Records — May to July 2026"
RANDOM_STATE = 42                                               # reproducibility
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function definitions

## Step 1 — Data selection & exploration

In [2]:
# Function for reading the monthly trip Parquet files
# Input: file_paths (list of str) - paths to the yellow taxi parquet files
# Output: a single pandas DataFrame containing all the trips of the period
def load_trips(file_paths):
    frames = []
    for p in file_paths:                                  # one DataFrame per month
        m = pd.read_parquet(p)
        for c in m.select_dtypes("float64").columns:      # downcast to halve memory (~10 M rows)
            m[c] = m[c].astype("float32")
        for c in m.select_dtypes("int64").columns:
            m[c] = m[c].astype("int32")
        frames.append(m)
    df = pd.concat(frames, ignore_index=True)             # stack the months vertically
    return df


# Function for reading the taxi zone reference data (names + polygons)
# Input: lookup_path (str) - path to taxi_zone_lookup.csv
#        shapefile_path (str) - path to taxi_zones.shp
# Output: (zones, zones_gdf) - DataFrame of zone names, GeoDataFrame of polygons in WGS84 (lat/lon)
def load_zones(lookup_path, shapefile_path):
    zones = pd.read_csv(lookup_path)                                   # LocationID, Borough, Zone, service_zone
    zones_gdf = gpd.read_file(shapefile_path).to_crs(epsg=4326)        # reproject from NY State Plane to lat/lon
    zones_gdf = zones_gdf.dissolve(by="LocationID", as_index=False)    # a few zones have duplicated rows
    zones_gdf["geometry"] = zones_gdf.geometry.simplify(0.0003)        # ~30 m tolerance: lighter maps, same shapes
    return zones, zones_gdf


# Function for summarising a DataFrame (structure, missing values, ranges)
# Input: df (DataFrame) - data to explore
# Output: summary (DataFrame) - one row per column: dtype, #missing, %missing, min, max, #unique
def explore_data(df):
    print(f"Number of rows    : {df.shape[0]:,}")
    print(f"Number of columns : {df.shape[1]}")
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),                                # data type of each column
        "missing": df.isna().sum(),                                    # number of missing values
        "missing_%": (df.isna().mean() * 100).round(2),                # share of missing values
        "min": df.min(numeric_only=False),                             # lower bound of the range
        "max": df.max(numeric_only=False),                             # upper bound of the range
        "n_unique": df.nunique(),                                      # cardinality
    })
    return summary

## Step 2 — Pre-processing & transformation (cleaning)

The exploration (see the outputs in the `__main__` section) shows several data-quality problems.
Decisions taken, and **why**:

| Removed rows / features | Reason |
|---|---|
| Pick-up outside 1 May – 31 July 2026 | A few taximeters have a wrong clock (dates in 2008, 2009, 2027…). They are outside the study period. |
| `trip_distance` ≤ 0 or > 100 miles | Zero-distance trips are cancelled rides; > 100 miles is not possible inside the TLC area in a normal trip. |
| Duration ≤ 1 min or > 3 h | Meter errors (forgotten meter, immediate cancellation). |
| `fare_amount` ≤ 0 or `total_amount` ≤ 0 or > $500 | Refunds / disputes are encoded as negative amounts; > $500 are typos. |
| Average speed > 80 mph | Physically impossible in NYC → inconsistent distance or timestamps. |
| `PULocationID` / `DOLocationID` = 264, 265 | "Unknown" / "Outside of NYC": they have no polygon, so they cannot be used in spatial analysis. |
| Columns `store_and_fwd_flag`, `VendorID`, `mta_tax`, `improvement_surcharge`, `extra` | Technical or constant fees that carry no analytical information for our questions. |
| Column `request_source` | New 2026 column, 83 % missing (only filled for app-requested trips): too incomplete to be used. |

`passenger_count`, `RatecodeID`, `congestion_surcharge`, `Airport_fee` have missing values for the same rows
(trips recorded by street-hail apps): we **keep** these rows (they are valid trips) and fill
`passenger_count` with 1 (the median) and the fees with 0.

New features created (transformation step):
- `duration_min` = (drop-off − pick-up) in minutes
- `speed_mph` = `trip_distance` / (`duration_min` / 60)
- `hour`, `weekday` (0 = Monday), `date`, `is_weekend`
- `tip_pct` = `tip_amount` / `fare_amount` × 100 (only meaningful for card payments)

In [3]:
# Function for cleaning the raw trips and creating derived features
# Input: df (DataFrame) - raw trips, start/end (Timestamp) - study period bounds
# Output: (clean_df, report) - cleaned DataFrame and a DataFrame logging rows removed at each rule
def clean_data(df, start, end):
    report = [("raw data", len(df))]                                   # keeps track of each filter
    # drop features without analytical value first (see markdown above) - also saves memory
    df = df.drop(columns=[c for c in ["store_and_fwd_flag", "VendorID", "mta_tax", "improvement_surcharge", "extra", "request_source"]
                          if c in df.columns])

    # derived feature needed by several filters
    df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

    rules = {                                                          # rule name -> boolean mask of rows to keep
        "pick-up inside study period": df["tpep_pickup_datetime"].between(start, end, inclusive="left"),
        "0 < distance <= 100 miles": df["trip_distance"].between(0, 100, inclusive="right"),
        "1 min < duration <= 3 h": df["duration_min"].between(1, 180, inclusive="right"),
        "0 < fare, 0 < total <= 500 $": (df["fare_amount"] > 0) & df["total_amount"].between(0, 500, inclusive="right"),
        "known zones (not 264/265)": df["PULocationID"].le(263) & df["DOLocationID"].le(263),
    }
    keep = pd.Series(True, index=df.index)
    for name, mask in rules.items():                                   # combine rules one by one to log their effect
        keep &= mask
        report.append((name, int(keep.sum())))
    df = df[keep]

    df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)
    df = df[df["speed_mph"] <= 80]                                     # impossible speeds
    report.append(("speed <= 80 mph", len(df)))

    # missing values: valid trips, fill with neutral values
    df["passenger_count"] = df["passenger_count"].fillna(1)
    for c in ["congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]:
        if c in df.columns:
            df[c] = df[c].fillna(0)
    df["RatecodeID"] = df["RatecodeID"].fillna(99)                     # 99 = unknown in the TLC dictionary

    # temporal features
    pu = df["tpep_pickup_datetime"]
    df["hour"] = pu.dt.hour
    df["weekday"] = pu.dt.weekday
    df["date"] = pu.dt.normalize()
    df["is_weekend"] = df["weekday"] >= 5
    df["tip_pct"] = df["tip_amount"] / df["fare_amount"] * 100

    report = pd.DataFrame(report, columns=["step", "rows_left"])
    report["rows_removed"] = (-report["rows_left"].diff()).fillna(0).astype(int)
    return df.reset_index(drop=True), report


# Function for attaching the zone / borough names to the pick-up and drop-off IDs
# Input: df (DataFrame) - clean trips, zones (DataFrame) - taxi zone lookup
# Output: DataFrame with PU_Borough, PU_Zone, DO_Borough, DO_Zone columns
def add_zone_names(df, zones):
    z = zones.set_index("LocationID")
    df["PU_Borough"] = df["PULocationID"].map(z["Borough"])
    df["PU_Zone"] = df["PULocationID"].map(z["Zone"])
    df["DO_Borough"] = df["DOLocationID"].map(z["Borough"])
    df["DO_Zone"] = df["DOLocationID"].map(z["Zone"])
    return df

## Step 3 — Data mining: the four queries

### Query 1 — Grouping: *Which pick-up zones generate the most revenue, and how generous are their customers?*

We group trips by pick-up zone (`PULocationID`) and compute, for each zone:

| Indicator | How it is calculated | Interpretation |
|---|---|---|
| **n_trips** | `count` of trips starting in the zone | Demand: how many customers start their trip here |
| **total_revenue** | `sum(total_amount)` (fare + tips + tolls + surcharges, $) | Economic weight of the zone for drivers |
| **avg_fare** | `mean(total_amount)` | Typical price of a trip starting here (high → long trips, e.g. airports) |
| **avg_tip_pct** | `mean(tip_amount / fare_amount × 100)` computed **on credit-card trips only** | Generosity of customers; cash tips are not recorded, so including cash trips would bias the mean downwards |
| **share_revenue_%** | `total_revenue / Σ total_revenue × 100` | Concentration: share of the whole city revenue made in this zone |

The result is also aggregated by borough. We show the **top 10 zones by total revenue**.

In [4]:
# Function computing revenue / demand / tip indicators per pick-up zone and per borough
# Input: df (DataFrame) - clean trips with zone names, top_n (int) - number of zones to return
# Output: (top_zones, by_borough) - DataFrames of indicators, sorted by total revenue
def query_grouping(df, top_n=10):
    card = df["payment_type"] == 1                                         # tips are only recorded for cards
    df = df.assign(tip_pct_card=df["tip_pct"].where(card))                 # NaN for non-card trips -> ignored by mean
    agg = dict(n_trips=("total_amount", "size"),
               total_revenue=("total_amount", "sum"),
               avg_fare=("total_amount", "mean"),
               avg_tip_pct=("tip_pct_card", "mean"))
    by_zone = df.groupby(["PU_Borough", "PU_Zone"]).agg(**agg).reset_index()
    by_zone["share_revenue_%"] = by_zone["total_revenue"] / by_zone["total_revenue"].sum() * 100
    top_zones = by_zone.sort_values("total_revenue", ascending=False).head(top_n)

    by_borough = df.groupby("PU_Borough").agg(**agg).reset_index()
    by_borough["share_revenue_%"] = by_borough["total_revenue"] / by_borough["total_revenue"].sum() * 100
    by_borough = by_borough.sort_values("total_revenue", ascending=False)
    return top_zones, by_borough


# Function plotting the result of query 1
# Input: top_zones (DataFrame) - output of query_grouping
# Output: plotly Figure (horizontal bar chart coloured by average tip %)
def plot_grouping(top_zones):
    fig = px.bar(top_zones.sort_values("total_revenue"), x="total_revenue", y="PU_Zone", orientation="h",
                 color="avg_tip_pct", color_continuous_scale="Blues",
                 hover_data=["PU_Borough", "n_trips", "avg_fare", "share_revenue_%"],
                 labels={"total_revenue": "Total revenue ($)", "PU_Zone": "Pick-up zone",
                         "avg_tip_pct": "Avg tip (%)"},
                 title="Q1 — Top 10 pick-up zones by total revenue (May–July 2026)")
    fig.update_layout(height=500)
    return fig

### Query 2 — Frequent pattern mining: *Which trip characteristics typically occur together?*

Each trip is turned into a **transaction** (a basket of items) by discretising its attributes:

| Item family | Values |
|---|---|
| `PU=` / `DO=` | pick-up / drop-off borough (Manhattan, Queens, …, `Airport` for JFK/LaGuardia/Newark zones) |
| `time=` | night (0–5h), morning (6–9h), midday (10–15h), evening (16–19h), late (20–23h) |
| `day=` | weekday / weekend |
| `pay=` | card / cash / flex (payment_type 0 = "flex fare", fixed price agreed in an app) / other |
| `dist=` | short (< 2 mi), medium (2–6 mi), long (≥ 6 mi) |
| `tip=` | none, low (< 15 %), standard (15–25 %), high (> 25 %) — card trips only |
| `pax=` | solo / group (≥ 2 passengers) |

We run the **Apriori** algorithm on a random sample of 300 000 trips (Apriori is memory-hungry on 10 M
rows; a sample of this size gives stable supports to ±0.2 %), then derive **association rules** `A → B`.

| Indicator | Formula | Interpretation |
|---|---|---|
| **support(A ∪ B)** | share of trips containing all items of A and B | How frequent the pattern is |
| **confidence(A → B)** | support(A ∪ B) / support(A) | Probability that a trip has B **given** it has A |
| **lift(A → B)** | confidence(A → B) / support(B) | > 1: A and B occur together more than by chance (positive association); = 1: independent; < 1: repel |

We keep rules with support ≥ 2 %, confidence ≥ 50 % and lift > 1.2, sorted by lift.

In [5]:
AIRPORT_ZONES = {1, 132, 138}   # Newark (EWR), JFK, LaGuardia


# Function converting each trip into a list of categorical items (one-hot encoded transactions)
# Input: df (DataFrame) - clean trips with zone names, sample_size (int), random_state (int)
# Output: boolean DataFrame (one column per item, one row per sampled trip), ready for apriori
def build_transactions(df, sample_size=300_000, random_state=RANDOM_STATE):
    s = df.sample(n=min(sample_size, len(df)), random_state=random_state)
    pu = np.where(s["PULocationID"].isin(AIRPORT_ZONES), "Airport", s["PU_Borough"])
    do = np.where(s["DOLocationID"].isin(AIRPORT_ZONES), "Airport", s["DO_Borough"])
    time = pd.cut(s["hour"], bins=[-1, 5, 9, 15, 19, 23],
                  labels=["night", "morning", "midday", "evening", "late"])
    pay = s["payment_type"].map({0: "flex", 1: "card", 2: "cash"}).fillna("other")
    dist = pd.cut(s["trip_distance"], bins=[0, 2, 6, np.inf], labels=["short", "medium", "long"], right=False)
    tip = pd.cut(s["tip_pct"], bins=[-np.inf, 0, 15, 25, np.inf], labels=["none", "low", "standard", "high"])
    tip = tip.astype(str).where(s["payment_type"] == 1, None)          # tip item only for card payments
    items = pd.DataFrame({
        "PU": pu, "DO": do, "time": time.astype(str),
        "day": np.where(s["is_weekend"], "weekend", "weekday"),
        "pay": pay, "dist": dist.astype(str), "tip": tip,
        "pax": np.where(s["passenger_count"] >= 2, "group", "solo"),
    })
    return pd.get_dummies(items, prefix_sep="=").astype(bool)           # one boolean column per item


# Function mining frequent itemsets (Apriori) and association rules
# Input: transactions (bool DataFrame), min_support, min_confidence, min_lift (float)
# Output: (itemsets, rules) - DataFrames; rules sorted by lift with readable antecedent/consequent strings
def frequent_patterns(transactions, min_support=0.02, min_confidence=0.5, min_lift=1.2):
    itemsets = apriori(transactions, min_support=min_support, use_colnames=True, max_len=4)
    rules = association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    rules = rules[rules["lift"] > min_lift].copy()
    rules["antecedent"] = rules["antecedents"].apply(lambda x: " & ".join(sorted(x)))
    rules["consequent"] = rules["consequents"].apply(lambda x: " & ".join(sorted(x)))
    rules = rules.sort_values("lift", ascending=False)
    return itemsets, rules[["antecedent", "consequent", "support", "confidence", "lift"]].reset_index(drop=True)


# Function plotting the association rules
# Input: rules (DataFrame) - output of frequent_patterns
# Output: plotly Figure (support vs confidence scatter, size and colour = lift)
def plot_rules(rules):
    fig = px.scatter(rules, x="support", y="confidence", size="lift", color="lift",
                     hover_data=["antecedent", "consequent"], color_continuous_scale="Viridis",
                     title="Q2 — Association rules (each point = one rule A → B)")
    fig.update_layout(height=500)
    return fig

### Query 3 — Temporal: *Can we forecast the hourly taxi demand?*

We build the **hourly time series** of the number of trips (pick-ups per hour, whole city).
The series has two strong seasonalities: **daily** (night low, evening peak) and **weekly** (weekdays vs weekends).

Method:
- **Training set**: 1 May – 30 June 2026; **test set**: July 2026 (744 hours), never seen by the model.
- **Model**: Holt-Winters exponential smoothing with additive seasonality of period 168 h (1 week, which
  contains the daily pattern), fitted on **log(1 + trips)**: on the log scale the night/day amplitude becomes
  proportional to the level, which matches the data much better. No trend component: over 2 months a trend
  extrapolated for a full month diverges (we tested it: MAPE > 40 %).
- The smoothing level α (how fast the level adapts to recent data) is chosen among {0.05, 0.1, 0.2, 0.3} on a
  **validation set** (last two weeks of June), then the model is refitted on all the training data.
- **Baseline**: *seasonal naïve* forecast = value observed at the same hour one week before
  (last week of June repeated). A useful model must beat this baseline.

| Indicator | Formula | Interpretation |
|---|---|---|
| **MAE** | mean(\|y − ŷ\|) | Average error in number of trips per hour |
| **RMSE** | √mean((y − ŷ)²) | Like MAE but penalises big misses (e.g. holidays) more |
| **MAPE** | mean(\|y − ŷ\| / y) × 100 | Average error in %, comparable between series |

July 4th (Independence Day) is expected to be badly predicted: holidays are not in the training seasonality.

In [6]:
# Function aggregating trips into an hourly count time series
# Input: df (DataFrame) - clean trips, start/end (Timestamp) - period bounds
# Output: pandas Series indexed by hour (continuous hourly index, missing hours = 0)
def hourly_series(df, start, end):
    s = df.set_index("tpep_pickup_datetime").resample("h").size()
    full_index = pd.date_range(start, end, freq="h", inclusive="left")
    return s.reindex(full_index, fill_value=0).rename("n_trips")


# Function training Holt-Winters on the train part and forecasting the test part
# Input: series (Series) - hourly counts, split_date (str/Timestamp) - first hour of the test set,
#        season (int) - seasonal period in hours, alphas (tuple) - candidate smoothing levels
# Output: (train, forecast_df, metrics) - DataFrame [actual, holt_winters, seasonal_naive] on the test period,
#         DataFrame of MAE / RMSE / MAPE for both methods
def forecast_demand(series, split_date="2026-07-01", season=168, alphas=(0.05, 0.1, 0.2, 0.3)):
    train, test = series[series.index < split_date], series[series.index >= split_date]

    # 1) choose the smoothing level alpha on a validation set = last 2 weeks of the training data
    #    (never on July, otherwise the test score would be optimistic)
    fit_part, valid = train.iloc[:-2 * season], train.iloc[-2 * season:]
    val_mae = {}
    for alpha in alphas:
        m = ExponentialSmoothing(np.log1p(fit_part), trend=None, seasonal="add", seasonal_periods=season)             .fit(smoothing_level=alpha)
        val_mae[alpha] = (valid - np.expm1(m.forecast(len(valid))).values).abs().mean()
    best_alpha = min(val_mae, key=val_mae.get)

    # 2) refit on the whole training set and forecast July (log scale -> back with expm1)
    model = ExponentialSmoothing(np.log1p(train), trend=None, seasonal="add", seasonal_periods=season)         .fit(smoothing_level=best_alpha)
    hw = pd.Series(np.expm1(model.forecast(len(test))).values, index=test.index).clip(lower=0)
    print(f"Validation MAE per alpha: { {a: round(v) for a, v in val_mae.items()} } -> alpha = {best_alpha}")

    last_week = train.iloc[-season:].values                               # seasonal naive baseline
    naive = pd.Series(np.resize(last_week, len(test)), index=test.index)

    forecast_df = pd.DataFrame({"actual": test, "holt_winters": hw, "seasonal_naive": naive})

    def metrics(y, yhat):
        err = y - yhat
        return {"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()),
                "MAPE_%": (err.abs() / y.replace(0, np.nan)).mean() * 100}
    metrics_df = pd.DataFrame({"Holt-Winters": metrics(test, hw), "Seasonal naive": metrics(test, naive)}).T
    return train, forecast_df, metrics_df


# Function plotting the history and the forecast
# Input: train (Series), forecast_df (DataFrame) - outputs of forecast_demand, history_days (int)
# Output: plotly Figure
def plot_forecast(train, forecast_df, history_days=14):
    hist = train.iloc[-history_days * 24:]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist.index, y=hist, name="train (actual)", line=dict(color="grey")))
    fig.add_trace(go.Scatter(x=forecast_df.index, y=forecast_df["actual"], name="July (actual)",
                             line=dict(color="black")))
    fig.add_trace(go.Scatter(x=forecast_df.index, y=forecast_df["holt_winters"], name="Holt-Winters forecast",
                             line=dict(color="#1f77b4")))
    fig.add_trace(go.Scatter(x=forecast_df.index, y=forecast_df["seasonal_naive"], name="Seasonal naive",
                             line=dict(color="#ff7f0e", dash="dot"), visible="legendonly"))
    fig.update_layout(title="Q3 — Hourly number of trips: July 2026 forecast vs actual",
                      xaxis_title="Hour", yaxis_title="Trips per hour", height=500)
    return fig

### Query 4 — Spatial clustering: *Which taxi zones behave alike?*

For every pick-up zone with at least 100 trips we compute a profile:

| Feature | Calculation | Meaning |
|---|---|---|
| `log_trips` | log10(number of pick-ups) | Demand intensity (log because it spans 4 orders of magnitude) |
| `avg_distance` | mean `trip_distance` (miles) | Typical trip length from this zone |
| `avg_fare` | mean `total_amount` ($) | Typical trip price |
| `night_share` | % of pick-ups between 20h and 5h | Night-life activity |
| `weekend_share` | % of pick-ups on Saturday/Sunday | Leisure vs business use |
| `avg_tip_pct` | mean tip % on card trips | Customer generosity |
| `lat`, `lon` | centroid of the zone polygon (shapefile) | Location, so that clusters are also geographically coherent |

All features are **standardised** (z-score) so that none dominates the Euclidean distance, then
**K-Means** groups the zones into *k* clusters. *k* is chosen by the **silhouette score**
(in [−1, 1]: how much closer a zone is to its own cluster than to the next one; higher is better),
tested for k = 3…8.

Result: a choropleth map of the clusters and a table of the average profile of each cluster,
used to name them (e.g. "Manhattan core", "airports", "outer boroughs residential"...).

In [7]:
# Function building one feature vector per pick-up zone (behaviour + location)
# Input: df (DataFrame) - clean trips, zones_gdf (GeoDataFrame) - zone polygons, min_trips (int)
# Output: DataFrame indexed by LocationID with the clustering features + zone names
def zone_features(df, zones_gdf, min_trips=100):
    d = df.assign(night=df["hour"].isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5]),
                  tip_card=df["tip_pct"].where(df["payment_type"] == 1))
    f = d.groupby("PULocationID").agg(n_trips=("fare_amount", "size"),
                                      avg_distance=("trip_distance", "mean"),
                                      avg_fare=("total_amount", "mean"),
                                      night_share=("night", "mean"),
                                      weekend_share=("is_weekend", "mean"),
                                      avg_tip_pct=("tip_card", "mean"))
    f = f[f["n_trips"] >= min_trips]                                   # too few trips -> unreliable profile
    f["log_trips"] = np.log10(f["n_trips"])
    f[["night_share", "weekend_share"]] *= 100
    cent = zones_gdf.set_index("LocationID").to_crs(epsg=2263).centroid.to_crs(epsg=4326)   # centroid in metric CRS
    f["lat"], f["lon"] = cent.y.reindex(f.index), cent.x.reindex(f.index)
    names = zones_gdf.set_index("LocationID")[["zone", "borough"]]
    f = f.join(names).dropna(subset=["lat", "avg_tip_pct"])
    return f


CLUSTER_FEATURES = ["log_trips", "avg_distance", "avg_fare", "night_share", "weekend_share", "avg_tip_pct", "lat", "lon"]


# Function choosing k with the silhouette score and running K-Means
# Input: features (DataFrame) - output of zone_features, k_range (iterable of int)
# Output: (features_with_cluster, silhouette_df, profile) - labelled zones, silhouette per k, mean profile per cluster
def cluster_zones(features, k_range=range(3, 9)):
    X = StandardScaler().fit_transform(features[CLUSTER_FEATURES])
    scores = {}
    for k in k_range:                                                   # model selection
        labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
        scores[k] = silhouette_score(X, labels)
    best_k = max(scores, key=scores.get)
    km = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit(X)
    features = features.assign(cluster=km.labels_.astype(str))
    silhouette_df = pd.DataFrame({"k": list(scores), "silhouette": list(scores.values())})
    profile = features.groupby("cluster")[["n_trips"] + CLUSTER_FEATURES[1:6]].mean()
    profile["n_zones"] = features["cluster"].value_counts()
    profile["example_zones"] = features.sort_values("n_trips", ascending=False).groupby("cluster")["zone"] \
        .apply(lambda z: ", ".join(z.head(3)))
    return features, silhouette_df, profile


# Function drawing the clusters on a map of NYC
# Input: features (DataFrame) - zones with cluster labels, zones_gdf (GeoDataFrame)
# Output: plotly Figure (choropleth map)
def plot_clusters(features, zones_gdf):
    g = zones_gdf.set_index("LocationID").join(features[["cluster", "n_trips", "avg_fare", "avg_tip_pct"]], how="inner")
    g = g.sort_values("cluster")
    fig = px.choropleth_map(g, geojson=g.geometry.__geo_interface__, locations=g.index, color="cluster",
                            hover_name="zone", hover_data=["borough", "n_trips", "avg_fare", "avg_tip_pct"],
                            map_style="carto-positron", center={"lat": 40.70, "lon": -73.94}, zoom=9.3,
                            opacity=0.65, title="Q4 — K-Means clusters of NYC taxi zones")
    fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0))
    return fig

### Bonus — External dataset: *Does the weather change taxi demand?*

**Second dataset:** NOAA *Global Historical Climatology Network – Daily* (GHCN-D), station **USW00094728 — New York
Central Park**, 1 May – 31 July 2026, downloaded from the NCEI Access Data Service (no account needed):
`https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00094728&startDate=2026-05-01&endDate=2026-07-31&dataTypes=PRCP,TMAX,TMIN,AWND&units=metric&format=csv`

| Column | Meaning |
|---|---|
| `PRCP` | Total precipitation of the day (mm) |
| `TMAX`, `TMIN` | Maximum / minimum temperature (°C) |
| `AWND` | Average wind speed (m/s) |

92 days, no missing values. It is joined to the taxi data on the **date** (temporal dimension).

**Method.** One row per day: number of taxi trips + weather. Daily demand depends strongly on the day of the week and
slowly decreases during the summer (students and residents leave the city), so a simple correlation would be
misleading. We fit a **multiple linear regression** on the logarithm of the daily number of trips:

`log(trips) = weekday effects + β_rain · rainy + β_temp · TMAX + β_hol · holiday + β_t · day_index + ε`

| Indicator | How it is calculated | Interpretation |
|---|---|---|
| **effect_%** | (exp(β) − 1) × 100 | % change in daily trips for one unit of the variable, **all other things equal** (same weekday, same period…) |
| `rainy` | 1 if `PRCP` ≥ 5 mm (a real rainy day, 17 days) | Effect of a rainy day vs a dry day |
| `TMAX` | per +1 °C | Effect of a warmer day |
| `holiday` | 1 on US federal holidays | Effect of a public holiday |
| `day_index` | per day elapsed since 1 May | Seasonal trend over the summer |
| **p-value** | Student t-test on β | < 0.05: the effect is statistically significant; otherwise it may be due to chance |
| **R²** | share of the day-to-day variance explained by the model | Close to 1 = the model explains demand well |

Finally, we use the weather to **explain the worst days of the Q3 forecast** (daily mean absolute error of
Holt-Winters, joined with precipitation and holidays).

In [8]:
# Function for reading the NOAA daily weather file
# Input: path (str) - CSV downloaded from the NCEI Access Data Service
# Output: DataFrame indexed by date with PRCP (mm), TMAX, TMIN (°C), AWND (m/s)
def load_weather(path):
    w = pd.read_csv(path, parse_dates=["DATE"])            # one row per day
    return w.drop(columns="STATION").set_index("DATE").sort_index()


# Function building the daily table taxi demand + weather + calendar
# Input: df (DataFrame) - clean trips, weather (DataFrame) - output of load_weather,
#        holidays (DatetimeIndex) - public holidays, rain_threshold (float, mm) - rainy day definition
# Output: DataFrame with one row per day
def daily_demand_weather(df, weather, holidays, rain_threshold=5.0):
    daily = df.groupby("date").size().rename("trips").to_frame()          # trips per day
    daily = daily.join(weather, how="inner")                              # temporal join on the date
    daily["weekday"] = daily.index.day_of_week
    daily["rainy"] = (daily["PRCP"] >= rain_threshold).astype(int)
    daily["holiday"] = daily.index.isin(holidays).astype(int)
    daily["day_index"] = np.arange(len(daily))                            # trend over the summer
    return daily


# Function estimating the effect of the weather on demand with a log-linear regression
# Input: daily (DataFrame) - output of daily_demand_weather
# Output: (model, effects) - fitted statsmodels OLS, DataFrame of % effects with 95% CI and p-values
def weather_regression(daily):
    model = smf.ols("np.log(trips) ~ C(weekday) + rainy + TMAX + holiday + day_index", data=daily).fit()
    terms = ["rainy", "TMAX", "holiday", "day_index"]
    ci = model.conf_int().loc[terms]
    effects = pd.DataFrame({
        "variable": ["rainy day (PRCP >= 5 mm)", "+1 °C of max temperature", "public holiday", "+1 day (summer trend)"],
        "effect_%": (np.exp(model.params[terms]) - 1).values * 100,       # log-model coefficient -> % change
        "ci_low_%": (np.exp(ci[0]) - 1).values * 100,
        "ci_high_%": (np.exp(ci[1]) - 1).values * 100,
        "p_value": model.pvalues[terms].values,
    })
    print(f"R² = {model.rsquared:.2f} on {int(model.nobs)} days")
    return model, effects


# Function joining the daily errors of the Q3 forecast with the weather and calendar
# Input: forecast_df (DataFrame) - output of forecast_demand, daily (DataFrame) - output of daily_demand_weather, top_n (int)
# Output: DataFrame of the top_n days with the largest mean absolute error
def forecast_errors_vs_weather(forecast_df, daily, top_n=6):
    err = (forecast_df["actual"] - forecast_df["holt_winters"])
    by_day = pd.DataFrame({"MAE": err.abs().groupby(err.index.normalize()).mean(),
                           "bias": err.groupby(err.index.normalize()).mean()})   # bias < 0: model over-forecast
    by_day = by_day.join(daily[["trips", "PRCP", "TMAX", "holiday"]])
    by_day.index = by_day.index.strftime("%a %d %b")
    return by_day.sort_values("MAE", ascending=False).head(top_n)


# Function plotting daily demand against the weather
# Input: daily (DataFrame) - output of daily_demand_weather
# Output: plotly Figure (daily trips over time, rain as bars on a second axis, holidays marked)
def plot_weather(daily):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=daily.index, y=daily["PRCP"], name="Precipitation (mm)", yaxis="y2",
                         marker_color="rgba(31,119,180,0.35)"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["trips"], name="Taxi trips per day", line=dict(color="black")))
    hol = daily[daily["holiday"] == 1]
    fig.add_trace(go.Scatter(x=hol.index, y=hol["trips"], mode="markers", name="Public holiday",
                             marker=dict(color="red", size=10, symbol="x")))
    fig.update_layout(title="Bonus — Daily taxi demand vs precipitation (NOAA Central Park)",
                      yaxis=dict(title="Trips per day"),
                      yaxis2=dict(title="Precipitation (mm)", overlaying="y", side="right", showgrid=False),
                      height=500, legend=dict(orientation="h", y=-0.15))
    return fig

## Step 4 — Interpretation & visualisation: the dashboard

The Dash application gathers the four queries and the weather bonus in five tabs, with the team members, the dataset name and
global KPIs (number of trips analysed, total revenue, average fare, average card tip).

Because a Dash server blocks the notebook, it is only started when `RUN_DASHBOARD = True` (default when the
notebook is run interactively). As **evidence that it works**, a static HTML export with the same figures is
written to `outputs/DIALLO_dashboard.html`.

In [9]:
# Function computing the global KPIs displayed at the top of the dashboard
# Input: df (DataFrame) - clean trips
# Output: dict {label: formatted value}
def compute_kpis(df):
    card = df["payment_type"] == 1
    return {"Trips analysed": f"{len(df):,}",
            "Total revenue": f"${df['total_amount'].sum() / 1e6:,.1f} M",
            "Average trip price": f"${df['total_amount'].mean():,.2f}",
            "Average card tip": f"{df.loc[card, 'tip_pct'].mean():.1f} %"}


# Function building the Dash application
# Input: figures (dict name -> plotly Figure), tables (dict name -> DataFrame), kpis (dict)
# Output: a dash.Dash app object (not started)
def build_dashboard(figures, tables, kpis):
    app = Dash(__name__)

    def table(df):                                                      # small HTML table helper
        df = df.apply(lambda col: col.astype("float64").round(2) if col.dtype.kind == "f" else col)  # float32 -> clean 2 decimals
        return html.Table([html.Tr([html.Th(c) for c in df.columns])] +
                          [html.Tr([html.Td(str(v)) for v in row]) for row in df.values],
                          style={"fontSize": "12px", "margin": "10px auto", "borderCollapse": "collapse"})

    kpi_cards = [html.Div([html.Div(v, style={"fontSize": "26px", "fontWeight": "bold"}), html.Div(k)],
                          style={"padding": "12px 24px", "background": "#f2f5fa", "borderRadius": "8px"})
                 for k, v in kpis.items()]

    app.layout = html.Div([
        html.H1("NYC Yellow Taxi — Data Science Dashboard", style={"textAlign": "center"}),
        html.H3(f"Dataset: {DATASET_NAME}", style={"textAlign": "center", "color": "#555"}),
        html.P("Team: " + ", ".join(TEAM), style={"textAlign": "center"}),
        html.Div(kpi_cards, style={"display": "flex", "gap": "16px", "justifyContent": "center"}),
        dcc.Tabs([
            dcc.Tab(label="Q1 — Revenue by zone", children=[dcc.Graph(figure=figures["q1"]), table(tables["q1"])]),
            dcc.Tab(label="Q2 — Frequent patterns", children=[dcc.Graph(figure=figures["q2"]), table(tables["q2"])]),
            dcc.Tab(label="Q3 — Demand forecast", children=[dcc.Graph(figure=figures["q3"]), table(tables["q3"])]),
            dcc.Tab(label="Q4 — Zone clustering", children=[dcc.Graph(figure=figures["q4"]), table(tables["q4"])]),
            dcc.Tab(label="Bonus — Weather (NOAA)", children=[dcc.Graph(figure=figures["q5"]), table(tables["q5"])]),
        ], style={"marginTop": "20px"}),
    ], style={"fontFamily": "Arial, sans-serif", "maxWidth": "1200px", "margin": "auto"})
    return app


# Function exporting the dashboard content to a standalone HTML file (evidence that it works)
# Input: figures (dict), tables (dict), kpis (dict), path (str) - output file
# Output: path of the written file
def export_dashboard_html(figures, tables, kpis, path):
    parts = [f"<h1>NYC Yellow Taxi — Data Science Dashboard</h1><h3>Dataset: {DATASET_NAME}</h3>",
             f"<p>Team: {', '.join(TEAM)}</p>",
             "<p>" + " &nbsp;|&nbsp; ".join(f"<b>{k}</b>: {v}" for k, v in kpis.items()) + "</p>"]
    for i, key in enumerate(["q1", "q2", "q3", "q4", "q5"]):
        parts.append(figures[key].to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
        parts.append(tables[key].round(2).to_html(index=False))
    with open(path, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>DIALLO dashboard</title></head>"
                "<body style='font-family:Arial;max-width:1200px;margin:auto'>" + "\n".join(parts) + "</body></html>")
    return path

# Main block

Runs the whole KDD pipeline in order. Results of each step are displayed below, followed by their interpretation.

In [10]:
if __name__ == "__main__":
    # Step 1: Data collection & exploration
    raw = load_trips(TRIP_FILES)
    zones, zones_gdf = load_zones(ZONE_LOOKUP, ZONE_SHAPEFILE)
    summary = explore_data(raw)
    display(summary)
    display(raw.head())

Number of rows    : 11,458,193
Number of columns : 21


,dtype,missing,missing_%,min,max,n_unique
VendorID,int32,0,0.00,1,7,4
tpep_pickup_datetime,datetime64[us],0,0.00,2008-12-30 23:06:00,2026-08-05 20:54:00,5389180
tpep_dropoff_datetime,datetime64[us],0,0.00,2008-12-30 23:15:45,2026-08-05 21:04:08,5389312
passenger_count,float32,2938598,25.65,0.00,9.00,10
trip_distance,float32,0,0.00,0.00,"318,129.09",6621
RatecodeID,float32,2938598,25.65,1.00,99.00,7
store_and_fwd_flag,str,2938598,25.65,N,Y,2
PULocationID,int32,0,0.00,1,265,261
DOLocationID,int32,0,0.00,1,265,262
payment_type,int32,0,0.00,0,5,6


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1.00,7.51,1.00,N,138,37,2,35.90,6.00,0.50,0.00,0.00,1.00,45.40,0.00,2.00,0.00,NaN
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1.00,6.14,1.00,N,138,237,1,27.50,6.00,0.50,9.56,0.00,1.00,49.81,2.50,2.00,0.75,NaN
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1.00,2.40,1.00,N,249,232,1,19.10,4.25,0.50,3.73,0.00,1.00,28.58,2.50,0.00,0.75,NaN
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0.00,1.20,1.00,N,232,114,1,9.30,4.25,0.50,3.00,0.00,1.00,18.05,2.50,0.00,0.75,NaN
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2.00,0.86,1.00,N,140,237,1,7.20,0.00,0.50,2.44,0.00,1.00,14.64,2.50,0.00,0.00,NaN


**Reading the exploration.** The table above gives, for each column, its type, the number and share of
missing values and its range. The minimum / maximum values reveal the anomalies handled in Step 2
(negative amounts, zero or huge distances, timestamps outside the period, location IDs 264/265).

In [11]:
if __name__ == "__main__":
    # Step 2: Cleaning & transformation
    df, cleaning_report = clean_data(raw, START, END)
    df = add_zone_names(df, zones)
    display(cleaning_report)
    print(f"Kept {len(df) / len(raw) * 100:.1f}% of the raw trips")
    del raw                                                         # free memory

,step,rows_left,rows_removed
0,raw data,11458193,0
1,pick-up inside study period,11458125,68
2,0 < distance <= 100 miles,11088211,369914
3,1 min < duration <= 3 h,10905772,182439
4,"0 < fare, 0 < total <= 500 $",10864495,41277
5,known zones (not 264/265),10794155,70340
6,speed <= 80 mph,10792394,1761


Kept 94.2% of the raw trips


In [12]:
if __name__ == "__main__":
    # Query 1: grouping
    q1_top, q1_borough = query_grouping(df)
    display(q1_top)
    display(q1_borough)
    fig_q1 = plot_grouping(q1_top)
    fig_q1.show()

,PU_Borough,PU_Zone,n_trips,total_revenue,avg_fare,avg_tip_pct,share_revenue_%
202,Queens,JFK Airport,404208,"31,341,292.00",77.54,19.42,9.70
209,Queens,LaGuardia Airport,280278,"19,681,872.00",70.22,24.77,6.09
143,Manhattan,Midtown Center,454677,"12,480,046.00",27.45,26.72,3.86
162,Manhattan,Upper East Side South,494318,"10,949,509.00",22.15,27.39,3.39
161,Manhattan,Upper East Side North,429171,"9,754,249.00",22.73,26.80,3.02
156,Manhattan,Times Sq/Theatre District,315181,"9,628,896.00",30.55,26.06,2.98
144,Manhattan,Midtown East,338175,"8,897,100.00",26.31,26.14,2.75
149,Manhattan,Penn Station/Madison Sq West,324415,"8,815,690.00",27.17,25.16,2.73
115,Manhattan,East Chelsea,280115,"7,909,840.50",28.24,25.81,2.45
135,Manhattan,Lincoln Square East,319005,"7,642,975.50",23.96,26.99,2.37


,PU_Borough,n_trips,total_revenue,avg_fare,avg_tip_pct,share_revenue_%
3,Manhattan,9487685,"247,810,608.00",26.12,25.92,76.73
4,Queens,897002,"60,336,228.00",67.26,21.08,18.68
1,Brooklyn,344746,"12,510,462.00",36.29,7.94,3.87
0,Bronx,62126,"2,251,844.25",36.25,1.36,0.70
5,Staten Island,773,"37,140.41",48.05,14.00,0.01
2,EWR,62,"7,362.51",118.75,16.03,0.00


**Interpretation Q1.**
- The two airports are by far the most lucrative pick-up zones: **JFK alone brings ~9.7 % of the whole city
  revenue and LaGuardia ~6 %**, with fewer trips than the busiest Manhattan zones but an average ticket of
  ~$70–78 (vs ~$22–30 in Manhattan). For a driver, one airport trip is worth about three Manhattan trips.
- At borough level, **Manhattan concentrates ~88 % of the trips and ~77 % of the revenue**: the yellow taxi is
  essentially a Manhattan + airports service. Brooklyn and the Bronx together are below 5 %.
- Card tips are around **25–27 % in Manhattan**, lower at JFK (~19 %, long fixed-fare trips to Manhattan).
  The very low card-tip rates in the Bronx/Brooklyn must be read with care: there, most trips are
  *flex-fare* app trips (payment_type 0, see Q2) and the remaining card payments are a biased subset.

In [13]:
if __name__ == "__main__":
    # Query 2: frequent pattern mining
    transactions = build_transactions(df)
    itemsets, q2_rules = frequent_patterns(transactions)
    print(f"{len(itemsets)} frequent itemsets, {len(q2_rules)} rules kept")
    display(q2_rules.head(15))
    fig_q2 = plot_rules(q2_rules)
    fig_q2.show()

1274 frequent itemsets, 896 rules kept


,antecedent,consequent,support,confidence,lift
0,dist=long & tip=standard,PU=Airport & pay=card,0.02,0.55,11.06
1,PU=Airport,DO=Manhattan & dist=long & pay=card,0.03,0.53,10.41
2,DO=Manhattan & dist=long & pay=card,PU=Airport,0.03,0.66,10.41
3,PU=Airport & tip=standard,dist=long & pay=card,0.02,0.96,10.39
4,DO=Manhattan & PU=Airport,dist=long & pay=card,0.03,0.88,9.61
5,dist=long & pay=card & tip=standard,PU=Airport,0.02,0.55,8.60
6,dist=long & tip=standard,PU=Airport,0.02,0.55,8.60
7,PU=Airport & pay=card,DO=Manhattan & dist=long,0.03,0.68,8.04
8,PU=Airport & day=weekday,dist=long & pay=card,0.03,0.74,8.02
9,PU=Airport,dist=long & pay=card,0.05,0.73,7.97


**Interpretation Q2.**
- The rules with the highest lift (≈ 8–11) all describe the **airport trip pattern**:
  `PU=Airport → DO=Manhattan & dist=long & pay=card`. A trip starting at an airport is ~10 times more likely
  than average to be a long card-paid trip to Manhattan. Airports are a clearly separate market.
- **Short Manhattan card trips come with high tips**: `dist=short & pay=card → tip=high` (support ≈ 27 %,
  confidence ≈ 72 %, lift ≈ 1.9). Customers tip proportionally more on small fares (the tip screen
  suggests fixed percentages / minimums), and this is reinforced in the evening and late evening.
- **Outer boroughs ↔ flex fare**: `PU=Brooklyn → pay=flex` (confidence ≈ 73 %, lift ≈ 3): outside Manhattan,
  yellow taxis are mostly booked through apps with an up-front price, not hailed in the street.
- **Night ↔ weekend in Manhattan**: `DO=Manhattan & time=night → PU=Manhattan & day=weekend` (lift ≈ 2):
  night trips are mostly week-end nightlife trips inside Manhattan.

In [14]:
if __name__ == "__main__":
    # Query 3: temporal forecasting
    series = hourly_series(df, START, END)
    train, q3_forecast, q3_metrics = forecast_demand(series)
    display(q3_metrics)
    fig_q3 = plot_forecast(train, q3_forecast)
    fig_q3.show()

Validation MAE per alpha: {0.05: 443, 0.1: 416, 0.2: 448, 0.3: 512} -> alpha = 0.1


,MAE,RMSE,MAPE_%
Holt-Winters,503.92,728.31,13.34
Seasonal naive,543.36,839.77,15.10


**Interpretation Q3.**
- Demand follows a very regular **weekly cycle**: low around 4–5 am, peak around 6 pm, Thursday/Friday the
  busiest days and Monday/Sunday the quietest.
- The log-scale Holt-Winters model forecasts a **whole month ahead** with a MAPE of ≈ 13 % (≈ 500 trips/hour
  of MAE), better than the seasonal-naive baseline (≈ 15 %), so the model captures the seasonality better than
  simply repeating the last week.
- The largest errors are around **4 July (Independence Day, 3–5 July)** and on the weekend of 18–19 July, where demand departs from the usual week:
  holidays are not in the training data. Adding a holiday calendar (or weather data, our planned
  external dataset) would be the next improvement.
- Practical use: the TLC or fleet operators can anticipate how many cabs are needed hour by hour.

In [15]:
if __name__ == "__main__":
    # Query 4: spatial clustering
    feats = zone_features(df, zones_gdf)
    q4_zones, q4_silhouette, q4_profile = cluster_zones(feats)
    display(q4_silhouette)
    display(q4_profile)
    fig_q4 = plot_clusters(q4_zones, zones_gdf)
    fig_q4.show()

,k,silhouette
0,3,0.30
1,4,0.31
2,5,0.34
3,6,0.35
4,7,0.34
5,8,0.33


,n_trips,avg_distance,avg_fare,night_share,weekend_share,avg_tip_pct,n_zones,example_zones
cluster,,,,,,,,
0,"23,384.09",5.15,36.76,48.66,44.27,14.83,46,"East Village, Greenwich Village South, Lower E..."
1,"2,494.41",7.38,36.74,25.45,33.49,2.61,49,"Washington Heights South, Washington Heights N..."
2,"170,886.76",2.93,27.00,29.16,28.64,24.31,51,"Upper East Side South, Midtown Center, Upper E..."
3,"2,777.89",6.72,35.66,21.73,27.03,2.23,38,"Crown Heights North, East New York, Prospect-L..."
4,"72,346.20",8.93,65.20,32.75,29.95,18.13,10,"JFK Airport, LaGuardia Airport, East Elmhurst"
5,"1,232.83",8.51,41.46,32.34,29.86,3.74,40,"Jamaica, Forest Hills, Richmond Hill"


**Interpretation Q4.** The silhouette score is highest for **k = 6** (≈ 0.35, a reasonable structure for
real-world data). Reading the cluster profiles table (cluster numbers are arbitrary labels):
- **Manhattan core / business** (Midtown, Upper East Side…): by far the highest volume, short trips (~3 mi),
  the highest card tips (~24 %).
- **Nightlife zones** (East Village, Greenwich Village, Lower East Side…): ~half of pick-ups at night and
  ~45 % on weekends.
- **Airports** (JFK, LaGuardia): long (~9 mi) and expensive (~$65) trips, medium volume.
- **Upper Manhattan / Bronx**, **Brooklyn residential**, **Queens residential**: low volume, longer trips,
  very low recorded card tips (flex-fare dominated). The coordinates make these three groups
  geographically coherent.

These clusters can guide where to station taxis (core and airports), when (nightlife zones on weekend nights)
and where app-based dispatch matters most (outer boroughs).

In [16]:
if __name__ == "__main__":
    # Bonus: external dataset (NOAA weather)
    weather = load_weather(WEATHER_FILE)
    display(explore_data(weather))
    daily = daily_demand_weather(df, weather, HOLIDAYS)
    weather_model, q5_effects = weather_regression(daily)
    display(q5_effects)
    q5_errors = forecast_errors_vs_weather(q3_forecast, daily)
    display(q5_errors)
    fig_q5 = plot_weather(daily)
    fig_q5.show()

Number of rows    : 92
Number of columns : 4


,dtype,missing,missing_%,min,max,n_unique
AWND,float64,0,0.00,0.60,4.20,30
PRCP,float64,0,0.00,0.00,41.10,28
TMAX,float64,0,0.00,13.30,37.80,36
TMIN,float64,0,0.00,7.20,27.80,28


R² = 0.68 on 92 days


,variable,effect_%,ci_low_%,ci_high_%,p_value
0,rainy day (PRCP >= 5 mm),2.81,-2.13,7.99,0.27
1,+1 °C of max temperature,0.51,0.06,0.95,0.03
2,public holiday,-27.78,-34.40,-20.50,0.00
3,+1 day (summer trend),-0.28,-0.36,-0.20,0.00


,MAE,bias,trips,PRCP,TMAX,holiday
Sat 04 Jul,"1,270.59","-1,270.59",80452,4.60,34.40,1
Fri 03 Jul,"1,092.25","-1,036.26",81974,3.30,36.70,1
Sun 19 Jul,926.13,844.86,115419,0.00,26.10,0
Sat 18 Jul,857.15,644.87,126423,41.10,24.40,0
Sun 05 Jul,769.32,-643.43,79700,3.30,27.20,0
Thu 02 Jul,766.37,-643.72,99224,0.00,37.80,0


**Interpretation (bonus).** The model explains **68 % of the day-to-day variance** of demand (R² = 0.68).
- **Public holidays are the strongest factor: −28 % of trips** (95 % CI −34 % to −21 %, p < 0.001). On Memorial Day,
  Juneteenth and the 4 July weekend, offices close and many New Yorkers leave the city.
- **Summer trend: −0.28 % per day**, i.e. about −23 % between early May and the end of July (end of the university
  year, holidays). This explains why July is lower than May–June in Q3.
- **Temperature: +0.5 % of trips per extra °C** (p = 0.03): on hot days people avoid walking and the subway.
- **Rain, on a daily scale, has no significant effect** (+2.8 %, CI −2 % to +8 %, p = 0.27). A daily total is too
  coarse: a short storm at 6 pm and a light drizzle all day give the same value. Hourly weather data would be
  needed to measure the "people grab a cab when it starts raining" effect (NOAA has not published 2026 hourly data yet).
- **Explaining the Q3 forecast errors**: the worst forecast days are **3, 4 and 5 July** (holiday weekend, the model
  over-forecasts by ~1 000 trips/hour, *bias < 0*) and 2 July (eve of the long weekend, 37.8 °C). The other big
  miss is the **18 July, the rainiest day of the period (41 mm)**, where demand was *higher* than forecast
  (+645 trips/hour on average). Adding a holiday indicator and the weather as external regressors is therefore the
  natural next step to improve the forecast.

In [17]:
if __name__ == "__main__":
    # Dashboard: export (evidence) + launch
    figures = {"q1": fig_q1, "q2": fig_q2, "q3": fig_q3, "q4": fig_q4, "q5": fig_q5}
    tables = {"q1": q1_top,
              "q2": q2_rules.head(15),
              "q3": q3_metrics.reset_index(names="method"),
              "q4": q4_profile.reset_index(),
              "q5": q5_effects}
    kpis = compute_kpis(df)
    print("Dashboard exported to", export_dashboard_html(figures, tables, kpis,
                                                         os.path.join(OUTPUT_DIR, "DIALLO_dashboard.html")))
    app = build_dashboard(figures, tables, kpis)
    RUN_DASHBOARD = os.environ.get("RUN_DASHBOARD", "1") == "1"     # set RUN_DASHBOARD=0 for headless runs
    if RUN_DASHBOARD:
        app.run(debug=False, port=8050)                             # open http://127.0.0.1:8050

Dashboard exported to outputs/DIALLO_dashboard.html
